# Part 1: Dataset Translation (English → Hindi, Malayalam, French)

This notebook translates all 32,431 sentences from the MAVEN event detection dataset into three target languages using Meta's **NLLB-200** (No Language Left Behind) translation model.

**Languages:** Hindi (`hin_Deva`), Malayalam (`mal_Mlym`), French (`fra_Latn`)  
**Model:** `facebook/nllb-200-distilled-600M`  
**Hardware:** NVIDIA Tesla T4 (Kaggle)

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/kishorer2k4/maven-dataset/valid.jsonl
/kaggle/input/datasets/kishorer2k4/maven-dataset/test.jsonl
/kaggle/input/datasets/kishorer2k4/maven-dataset/train.jsonl


## 1. Load and Explore MAVEN Dataset

In [2]:
import json

TRAIN_FILE = "/kaggle/input/datasets/kishorer2k4/maven-dataset/train.jsonl"

with open(TRAIN_FILE, "r") as f:
    sample = json.loads(next(f))

print(sample.keys())

dict_keys(['title', 'id', 'content', 'events', 'negative_triggers'])


In [3]:
for k, v in sample.items():
    print(f"\n{k}")
    print(type(v))

    if isinstance(v, list):
        print("Length:", len(v))

        if len(v) > 0:
            print("First item:")
            print(v[0])

    else:
        print(v)


title
<class 'str'>
2006 Pangandaran earthquake and tsunami

id
<class 'str'>
8307a6b61b84d4eea42c1dd5e6e2cdba

content
<class 'list'>
Length: 9
First item:
{'sentence': 'The 2006 Pangandaran earthquake and tsunami occurred on July 17 at along a subduction zone off the coast of west and central Java, a large and densely populated island in the Indonesian archipelago.', 'tokens': ['The', '2006', 'Pangandaran', 'earthquake', 'and', 'tsunami', 'occurred', 'on', 'July', '17', 'at', 'along', 'a', 'subduction', 'zone', 'off', 'the', 'coast', 'of', 'west', 'and', 'central', 'Java', ',', 'a', 'large', 'and', 'densely', 'populated', 'island', 'in', 'the', 'Indonesian', 'archipelago', '.']}

events
<class 'list'>
Length: 39
First item:
{'id': '40b3b20bc2eeb6b163538b82c1379ead', 'type': 'Know', 'type_id': 1, 'mention': [{'trigger_word': 'observed', 'sent_id': 5, 'offset': [12, 13], 'id': '7fcf445a679aa13511278d321a908bd2'}]}

negative_triggers
<class 'list'>
Length: 117
First item:
{'trigger_wor

In [4]:
import json
from tqdm.auto import tqdm

docs = 0
sentences = 0

with open(TRAIN_FILE) as f:
    for line in tqdm(f):
        item = json.loads(line)

        docs += 1
        sentences += len(item["content"])

print("Documents :", docs)
print("Sentences :", sentences)

0it [00:00, ?it/s]

Documents : 2913
Sentences : 32431


In [5]:
import json
from tqdm.auto import tqdm

TRAIN_FILE = "/kaggle/input/datasets/kishorer2k4/maven-dataset/train.jsonl"

sentences = []

with open(TRAIN_FILE) as f:
    for line in tqdm(f):
        doc = json.loads(line)

        for sent in doc["content"]:
            sentences.append(sent["sentence"])

print("Total sentences:", len(sentences))

print("\nSample:")
print(sentences[0])

0it [00:00, ?it/s]

Total sentences: 32431

Sample:
The 2006 Pangandaran earthquake and tsunami occurred on July 17 at along a subduction zone off the coast of west and central Java, a large and densely populated island in the Indonesian archipelago.


In [6]:
lengths = [len(s.split()) for s in sentences]

print("Average words:", sum(lengths)/len(lengths))
print("Max words:", max(lengths))
print("Min words:", min(lengths))

Average words: 22.63877154574327
Max words: 267
Min words: 1


In [8]:
total_words = sum(lengths)

print(f"Total words: {total_words:,}")
print(f"Approx million words: {total_words/1_000_000:.2f}")

Total words: 734,198
Approx million words: 0.73


## 2. Setup: Install Dependencies & Load Translation Model

In [1]:
!pip install -q transformers sentencepiece accelerate sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 2.5 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 98.1 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is inc

In [2]:
import transformers
import torch

print(transformers.__version__)
print(torch.__version__)
print(torch.cuda.is_available())

5.0.0
2.10.0+cu128
True


In [3]:
!nvidia-smi

Wed Jun  3 08:14:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(MODEL)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

print("Model loaded successfully")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded successfully


In [5]:
import time
import torch

def translate_batch(texts, target_lang, batch_size=16):

    tokenizer.src_lang = "eng_Latn"

    outputs = []

    for i in range(0, len(texts), batch_size):

        batch = texts[i:i+batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt"
        ).to(model.device)

        generated = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(target_lang),
            max_new_tokens=256
        )

        outputs.extend(
            tokenizer.batch_decode(
                generated,
                skip_special_tokens=True
            )
        )

    return outputs

## 3. Test Translation (Sample Batch)

In [6]:
sample = sentences[:10]

start = time.time()

translated = translate_batch(
    sample,
    target_lang="hin_Deva"
)

elapsed = time.time() - start

print(f"Time: {elapsed:.2f} sec")

for i in range(3):
    print("\nEN :", sample[i])
    print("HI :", translated[i])

NameError: name 'sentences' is not defined

In [7]:
import json
from tqdm.auto import tqdm

TRAIN_FILE = "/kaggle/input/datasets/kishorer2k4/maven-dataset/train.jsonl"

sentences = []

with open(TRAIN_FILE) as f:
    for line in tqdm(f):
        doc = json.loads(line)

        for sent in doc["content"]:
            sentences.append(sent["sentence"])

print("Loaded", len(sentences), "sentences")

0it [00:00, ?it/s]

Loaded 32431 sentences


In [8]:
sample = sentences[:10]

import time

start = time.time()

translated = translate_batch(
    sample,
    target_lang="hin_Deva"
)

elapsed = time.time() - start

print("Time:", elapsed)

for i in range(3):
    print("\nEN:", sample[i])
    print("HI:", translated[i])

Time: 54.54017376899719

EN: The 2006 Pangandaran earthquake and tsunami occurred on July 17 at along a subduction zone off the coast of west and central Java, a large and densely populated island in the Indonesian archipelago.
HI: 2006 के पंगंदारन भूकंप और सुनामी 17 जुलाई को पश्चिमी और मध्य जावा के तट के एक सबडक्शन जोन के साथ हुआ था, जो इंडोनेशियाई द्वीपसमूह में एक बड़ा और घना आबादी वाला द्वीप है।

EN: The shock had a moment magnitude of 7.7 and a maximum perceived intensity of IV ("Light") in Jakarta, the capital and largest city of Indonesia.
HI: झटका का क्षण 7.7 और अधिकतम कथित तीव्रता IV ("प्रकाश") जकार्ता में था, जो इंडोनेशिया की राजधानी और सबसे बड़ा शहर है।

EN: There were no direct effects of the earthquake's shaking due to its low intensity, and the large loss of life from the event was due to the resulting tsunami, which inundated a portion of the Java coast that had been unaffected by the earlier 2004 Indian Ocean earthquake and tsunami that was off the coast of Sumatra.
HI: 

In [9]:
import torch

print("CUDA:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))
print("Model device:", next(model.parameters()).device)

CUDA: True
Device: Tesla T4
Model device: cpu


In [11]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

print(next(model.parameters()).device)

cuda:0


In [12]:
sample = sentences[:128]

start = time.time()

translated = translate_batch(
    sample,
    target_lang="hin_Deva",
    batch_size=128
)

elapsed = time.time() - start

print("Sentences:", len(sample))
print("Time:", elapsed)
print("Sent/sec:", len(sample)/elapsed)

Sentences: 128
Time: 5.168988227844238
Sent/sec: 24.763066650160138


## 4. Full Dataset Translation Pipeline

In [13]:
import json
import os
from tqdm.auto import tqdm

def translate_dataset(
    sentences,
    target_lang,
    output_file,
    checkpoint_every=1000,
    batch_size=128
):

    translated = []

    if os.path.exists(output_file):
        with open(output_file, "r", encoding="utf-8") as f:
            translated = json.load(f)

        print("Resuming from", len(translated))

    start_idx = len(translated)

    for i in tqdm(range(start_idx, len(sentences), batch_size)):

        batch = sentences[i:i+batch_size]

        results = translate_batch(
            batch,
            target_lang=target_lang,
            batch_size=batch_size
        )

        translated.extend(results)

        if len(translated) % checkpoint_every < batch_size:

            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(
                    translated,
                    f,
                    ensure_ascii=False
                )

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(
            translated,
            f,
            ensure_ascii=False
        )

    return translated

### 4.1 Hindi Translation

In [14]:
hindi_sentences = translate_dataset(
    sentences,
    target_lang="hin_Deva",
    output_file="/kaggle/working/maven_hindi.json",
    batch_size=128
)

  0%|          | 0/254 [00:00<?, ?it/s]

In [15]:
print(len(hindi_sentences))
print(hindi_sentences[:3])

32431
['2006 के पंगंदारन भूकंप और सुनामी 17 जुलाई को पश्चिमी और मध्य जावा के तट के एक सबडक्शन जोन के साथ हुआ था, जो इंडोनेशियाई द्वीपसमूह में एक बड़ा और घना आबादी वाला द्वीप है।', 'झटका का क्षण 7.7 और अधिकतम कथित तीव्रता IV ("प्रकाश") जकार्ता में था, जो इंडोनेशिया की राजधानी और सबसे बड़ा शहर है।', 'भूकंप के झटके के कारण इसकी कम तीव्रता के कारण कोई प्रत्यक्ष प्रभाव नहीं पड़ा, और घटना से होने वाली बड़ी जान-माल की वजह से हुई त्सुनामी थी, जिसने जावा तट के उस हिस्से को बाढ़ दी, जो 2004 के पहले हिंद महासागर में भूकंप और त्सुनामी से प्रभावित नहीं हुआ था।']


### 4.2 Verify Hindi Translations

In [16]:
import json

TRAIN_FILE = "/kaggle/input/datasets/kishorer2k4/maven-dataset/train.jsonl"

with open(TRAIN_FILE) as f:
    doc = json.loads(next(f))

print("Title:", doc["title"])

for ev in doc["events"][:5]:
    mention = ev["mention"][0]

    sid = mention["sent_id"]

    print("\nTYPE:", ev["type"])
    print("TRIGGER:", mention["trigger_word"])

    print("ENGLISH SENTENCE:")
    print(doc["content"][sid]["sentence"])

    print("\nHINDI SENTENCE:")
    print(hindi_sentences[sid])

Title: 2006 Pangandaran earthquake and tsunami

TYPE: Know
TRIGGER: observed
ENGLISH SENTENCE:
Several thousand kilometers to the southeast, surges of several meters were observed in northwestern Australia, but in Java the tsunami runups (height above normal sea level) were typically and resulted in the deaths of more than 600 people.

HINDI SENTENCE:
दक्षिण-पूर्व में कई हज़ार किलोमीटर की दूरी पर, उत्तर-पश्चिमी ऑस्ट्रेलिया में कई मीटर की तरंगें देखी गई, लेकिन जावा में सुनामी रनअप (सामान्य समुद्र तल से ऊपर की ऊंचाई) आम तौर पर होती थीं और इससे 600 से अधिक लोगों की मौत हो गई।

TYPE: Warning
TRIGGER: warning
ENGLISH SENTENCE:
Since the shock was felt with only moderate intensity well inland, and even less so at the shore, the surge arrived with little or no warning.

HINDI SENTENCE:
चूंकि आघात को केवल मध्यम तीव्रता के साथ महसूस किया गया था, तो यह भी कम है कि यह भूमि के अंदर और तट पर था, इसलिए लहर कम या कोई चेतावनी के साथ आई।

TYPE: Catastrophe
TRIGGER: earthquake
ENGLISH SENTENCE:
There we

In [17]:
import json
from collections import Counter

TRAIN_FILE = "/kaggle/input/datasets/kishorer2k4/maven-dataset/train.jsonl"

trigger_counter = Counter()

with open(TRAIN_FILE) as f:
    for line in f:
        doc = json.loads(line)

        for event in doc["events"]:
            for mention in event["mention"]:
                trigger_counter[mention["trigger_word"].lower()] += 1

print("Unique triggers:", len(trigger_counter))
print("\nTop 50 triggers:\n")

for trigger, count in trigger_counter.most_common(50):
    print(trigger, count)

Unique triggers: 6144

Top 50 triggers:

killed 828
battle 780
storm 776
began 770
hurricane 690
war 667
held 655
caused 607
attack 587
took place 586
damage 563
occurred 536
tour 536
became 499
fought 390
tournament 360
festival 348
destroyed 330
ended 316
captured 312
made 312
support 308
played 306
forces 303
reported 297
defeated 297
died 294
called 287
moved 285
reached 283
cyclone 276
attacks 274
used 272
found 267
conflict 263
control 258
match 258
included 252
weakened 252
final 249
resulted in 243
produced 241
damaged 239
attacked 236
developed 231
named 227
lost 226
death 222
event 220
formed 219


In [18]:
import json
from collections import Counter

TRAIN_FILE = "/kaggle/input/datasets/kishorer2k4/maven-dataset/train.jsonl"

event_counter = Counter()

with open(TRAIN_FILE) as f:
    for line in f:
        doc = json.loads(line)

        for event in doc["events"]:
            event_counter[event["type"]] += len(event["mention"])

print("Number of event types:", len(event_counter))
print()

for event, count in event_counter.most_common(30):
    print(f"{event:25} {count}")

Number of event types: 168

Catastrophe               3145
Attack                    2920
Hostile_encounter         2856
Causation                 2728
Process_start             2628
Competition               2460
Motion                    2165
Social_event              1653
Killing                   1625
Conquering                1432
Statement                 1392
Process_end               1392
Self_motion               1282
Arriving                  1236
Destroying                1176
Coming_to_be              1120
Bodily_harm               1024
Military_operation        1022
Death                     1001
Damaging                  981
Cause_change_of_strength  948
Creating                  936
Hold                      922
Control                   911
Cause_change_of_position_on_a_scale 903
Earnings_and_losses       891
Becoming                  848
Getting                   842
Arranging                 817
Know                      810


In [19]:
from collections import Counter
import json

event_counter = Counter()

with open(TRAIN_FILE) as f:
    for line in f:
        doc = json.loads(line)

        for event in doc["events"]:
            event_counter[event["type"]] += len(event["mention"])

counts = list(event_counter.values())

print("Classes:", len(counts))
print("Max:", max(counts))
print("Min:", min(counts))
print("Median:", sorted(counts)[len(counts)//2])

print("\nBottom 20 classes:\n")

for cls, cnt in sorted(event_counter.items(), key=lambda x: x[1])[:20]:
    print(cls, cnt)

Classes: 168
Max: 3145
Min: 4
Median: 242

Bottom 20 classes:

Change_tool 4
Breathing 6
Emergency 7
Extradition 11
Institutionalization 11
Risk 12
Incident 15
Ingestion 16
Renting 17
Theft 19
Vocalizations 20
Suspicion 22
Ratification 27
Justifying 27
Scouring 32
Labeling 35
Lighting 36
Practice 37
Submitting_documents 47
Carry_goods 48


In [22]:
import random

indices = random.sample(range(len(sentences)), 5)

for idx in indices:
    print("="*80)
    print("EN:")
    print(sentences[idx])

    print("\nHI:")
    print(hindi_sentences[idx])

EN:
The war continued in low intensity throughout the 1980s, though Morocco made several attempts to take the upper hand in 1989–1991.

HI:
1980 के दशक में युद्ध कम तीव्रता में जारी रहा, हालांकि मोरक्को ने 19891991 में ऊपर का हाथ लेने के कई प्रयास किए।
EN:
The attacks resulted in the paralysis of Yugoslav civilian and military command and control, the widespread destruction of Belgrade's infrastructure, and many civilian casualties.

HI:
इन हमलों के परिणामस्वरूप यूगोस्लाव नागरिक और सैन्य कमांड और नियंत्रण को पक्षाघात हो गया, बेलग्रेड के बुनियादी ढांचे का व्यापक विनाश हो गया, और कई नागरिकों की मौत हो गई।
EN:
Severe Tropical Cyclone Ron was the strongest tropical cyclone on record to impact Tonga.

HI:
गंभीर उष्णकटिबंधीय चक्रवात रोन टोंगा को प्रभावित करने के लिए रिकॉर्ड में सबसे मजबूत उष्णकटिबंधीय चक्रवात था।
EN:
The Kill to Get Crimson Tour was a 2008 concert tour by British singer-songwriter and guitarist Mark Knopfler, promoting the release of his album "Kill to Get Crimson".

HI:
द क

### 4.3 Malayalam Translation

In [23]:
malayalam_sentences = translate_dataset(
    sentences,
    target_lang="mal_Mlym",
    output_file="/kaggle/working/maven_malayalam.json",
    batch_size=128
)

Resuming from 1024


  0%|          | 0/246 [00:00<?, ?it/s]

### 4.4 French Translation

In [24]:
french_sentences = translate_dataset(
    sentences,
    target_lang="fra_Latn",
    output_file="/kaggle/working/maven_french.json",
    batch_size=128
)

  0%|          | 0/254 [00:00<?, ?it/s]

### 4.5 Verify All Translations

In [25]:
import json

for file in [
    "/kaggle/working/maven_hindi.json",
    "/kaggle/working/maven_malayalam.json",
    "/kaggle/working/maven_french.json"
]:
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(file, len(data))

/kaggle/working/maven_hindi.json 32431
/kaggle/working/maven_malayalam.json 32431
/kaggle/working/maven_french.json 32431


In [26]:
import random

indices = random.sample(range(len(sentences)), 5)

for idx in indices:
    print("="*80)
    print("EN:")
    print(sentences[idx])

    print("\nML:")
    print(malayalam_sentences[idx])

EN:
Becoming a hurricane on September 19, its strengthening rate increased while passing south of Jamaica.

ML:
സെപ്റ്റംബർ 19 ന് ചുഴലിക്കാറ്റ് ആയി മാറിക്കൊണ്ടിരിക്കുന്ന ഈ ചുഴലിക്കാറ്റ് ജമൈക്കയുടെ തെക്ക് ഭാഗത്ത് കടന്നുപോകുമ്പോൾ ശക്തിപ്പെടുത്തുന്ന വേഗത വർദ്ധിച്ചു.
EN:
In Lika, two guard brigades quickly cut the ARSK-held area (which lacked tactical depth and mobile reserve forces), isolating pockets of resistance, positioning a mobile force for a decisive northward thrust into the Karlovac Corps area of responsibility (AOR), and pushing ARSK towards Banovina.

ML:
ലിക്കയിൽ, രണ്ട് ഗാർഡ് ബ്രിഗഡുകൾ വേഗത്തിൽ ആർഎസ്എസ്കിയുടെ നിയന്ത്രണത്തിലുള്ള മേഖലയെ (താക്റ്റിക്കൽ ആഴവും മൊബൈൽ റിസർവ് ഫോഴ്സുകളും ഇല്ലാത്തത്) മുറിച്ചുമാറ്റി, പ്രതിരോധത്തിന്റെ പോക്കറ്റുകൾ ഒറ്റപ്പെടുത്തുകയും, നിർണായകമായി വടക്കോട്ട് കുതിക്കാൻ ഒരു മൊബൈൽ ഫോഴ്സ് സ്ഥാനം നൽകുകയും, കാർലോവാക് കോർപ്സ് ഉത്തരവാദിത്ത മേഖലയിലേക്ക് (AOR) ARSK-നെ ബാനോവിനയിലേക്കുള്ള വഴിക്ക് തള്ളുകയും ചെയ്തു.
EN:
The earthquake was especially destructive in the epice